In [1]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import torch
from torch.autograd import Variable
import numpy as np
import torch.functional as F
import torch.nn.functional as F

In [3]:
corpus = [
    'he is a king',
    'she is a queen',
    'he is a man',
    'she is a woman',
    'warsaw is poland capital',
    'berlin is germany capital',
    'paris is france capital',   
]

In [17]:
def tokenize_corpus(corpus):
    tokens = [x.split() for x in corpus]
    return tokens

tokenized_corpus = tokenize_corpus(corpus)
print(tokenized_corpus)

[['he', 'is', 'a', 'king'], ['she', 'is', 'a', 'queen'], ['he', 'is', 'a', 'man'], ['she', 'is', 'a', 'woman'], ['warsaw', 'is', 'poland', 'capital'], ['berlin', 'is', 'germany', 'capital'], ['paris', 'is', 'france', 'capital']]


In [5]:
vocabulary = []
for sentence in tokenized_corpus:
    for token in sentence:
        if token not in vocabulary:
            vocabulary.append(token)

word2idx = {w: idx for (idx, w) in enumerate(vocabulary)}
idx2word = {idx: w for (idx, w) in enumerate(vocabulary)}

vocabulary_size = len(vocabulary)

In [19]:
window_size = 2
idx_pairs = []
# for each sentence
for sentence in tokenized_corpus:
    indices = [word2idx[word] for word in sentence]
    # for each word, threated as center word
    for center_word_pos in range(len(indices)):
        # for each window position
        for w in range(-window_size, window_size + 1):
            context_word_pos = center_word_pos + w
            # make soure not jump out sentence
            if context_word_pos < 0 or context_word_pos >= len(indices) or center_word_pos == context_word_pos:
                continue
            context_word_idx = indices[context_word_pos]
            idx_pairs.append((indices[center_word_pos], context_word_idx))

idx_pairs = np.array(idx_pairs) # it will be useful to have this as numpy array

In [7]:
def get_input_layer(word_idx):
    x = torch.zeros(vocabulary_size).float()
    x[word_idx] = 1.0
    return x

In [59]:
# Adding so that results are always the same
torch.manual_seed(0)
np.random.seed(0)

embedding_dims = 8
num_epochs = 201
learning_rate = 0.001

def w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size, print_loss=True):
    """Train a Word2Vec model using the Skip-Gram approach."""
    W1 = Variable(torch.randn(embedding_dims, vocabulary_size).float(), requires_grad=True)
    W2 = Variable(torch.randn(vocabulary_size, embedding_dims).float(), requires_grad=True)
    for epo in range(num_epochs):
        loss_val = 0
        for data, target in idx_pairs:
            x = Variable(get_input_layer(data)).float()
            y_true = Variable(torch.from_numpy(np.array([target])).long())

            z1 = torch.matmul(W1, x)
            z2 = torch.matmul(W2, z1)

            log_softmax = F.log_softmax(z2, dim=0)

            loss = F.nll_loss(log_softmax.view(1,-1), y_true)
            # print(data, loss.data[0])
            loss_val += loss.item()

            loss.backward()
            W1.data -= learning_rate * W1.grad.data
            W2.data -= learning_rate * W2.grad.data

            W1.grad.data.zero_()
            W2.grad.data.zero_()
        if print_loss and epo % 10 == 0:
            print(f'Loss at epo {epo}: {loss_val/len(idx_pairs)}')
    return W1, W2

W1, W2 = w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size)

def similarity(v,u):
    return torch.dot(v,u)/(torch.norm(v)*torch.norm(u))

Loss at epo 0: 6.3228838255628945
Loss at epo 10: 5.047896347274738
Loss at epo 20: 4.436945485855852
Loss at epo 30: 4.04438270841326
Loss at epo 40: 3.757607816904783
Loss at epo 50: 3.5313432836106844
Loss at epo 60: 3.3476371837513788
Loss at epo 70: 3.1966816674385754
Loss at epo 80: 3.0703236814056125
Loss at epo 90: 2.961810506241662
Loss at epo 100: 2.866376081109047
Loss at epo 110: 2.780951634475163
Loss at epo 120: 2.7035798196281706
Loss at epo 130: 2.63297150858811
Loss at epo 140: 2.5682277011019843
Loss at epo 150: 2.508672400883266
Loss at epo 160: 2.4537481955119542
Loss at epo 170: 2.402965938619205
Loss at epo 180: 2.355892213327544
Loss at epo 190: 2.3121508125747954
Loss at epo 200: 2.271422046422958


In [43]:
print(vocabulary_size)

15


In [60]:
word_pairs = [
    ("she", "king"),
    ("she", "queen"),
    ("he", "king"),
    ("he", "queen"),
    ("warsaw", "berlin"),
    ("warsaw", "paris"),
    ("berlin", "paris"),]

def score_word_pairs(word_pairs):
    for word1, word2 in word_pairs:
        s = similarity(W2[word2idx[word1]], W2[word2idx[word2]])
        print(f'SIMILARITY {word1} - {word2}: {s.item()}')
score_word_pairs(word_pairs)

SIMILARITY she - king: 0.1672080010175705
SIMILARITY she - queen: -0.03226333484053612
SIMILARITY he - king: 0.4397442936897278
SIMILARITY he - queen: -0.1406475305557251
SIMILARITY warsaw - berlin: 0.057587623596191406
SIMILARITY warsaw - paris: 0.6335696578025818
SIMILARITY berlin - paris: -0.3479728102684021


In [35]:
score_word_pairs([("she", "king"),("she", "queen")])

SIMILARITY she - king: 0.27248474955558777
SIMILARITY she - queen: -0.05312594398856163


<b>Question 1:</b><p>
```
similarity("she”, "king") = ?
similarity("She", "queen") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
King has a much lower, negative score than queen, which is expected, as king and queen are more similar than she and king.<p>
The reason why there is not a strong similarity between 'she' and 'queen' is that 'she' and 'queen' have two words in between them (see cell 4 where I print out the tokens).<p>

In [36]:
score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])

SIMILARITY warsaw - poland: 0.47337985038757324
SIMILARITY warsaw - germany: -0.6303616762161255


<b>Question 2:</b><p>
```
similarity("warsaw", "poland") = ?
similarity("warsaw", "germany") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
Something is WRONG

In [37]:
score_word_pairs([("warsaw", "capital"),("poland", "capital")])

SIMILARITY warsaw - capital: 0.020551471039652824
SIMILARITY poland - capital: 0.3362537622451782


<b>Question 3:</b><p>
```
similarity("warsaw", "capital") = ?
similarity("poland", "capital") = ?
```
<b>Which pair is more similar? Does the model match your expectations?</b><p>
This is the first match where it is exactly as I expected. Warsaw is the capital of Poland, so they are very similar.<p>

In [42]:
embedding_dims = 8
num_epochs = 201
W1, W2 = w2v_train(embedding_dims, num_epochs, learning_rate, vocabulary_size, False)
score_word_pairs([("she", "king"),("she", "queen")])
score_word_pairs([("warsaw", "poland"),("warsaw", "germany")])
score_word_pairs([("warsaw", "capital"),("poland", "capital")])

SIMILARITY she - king: 0.2155856192111969
SIMILARITY she - queen: 0.3446895182132721
SIMILARITY warsaw - poland: -0.5156630873680115
SIMILARITY warsaw - germany: 0.04547814652323723
SIMILARITY warsaw - capital: 0.0713941752910614
SIMILARITY poland - capital: -0.007667593192309141


<b>Question 4:</b><p>
<b>Retrain the model with embedding_dims = 8 and epochs = 201 and check the pairs above again.<p>
Does the model seem to do better, worse, or about the same? Why?</b><p>


<b>Question 5</b><p>
Add your own sentences to the corpus, retrain, and test the similarity relationships.

Does the model do what you would expect?